<a href="https://colab.research.google.com/github/J053n1l50n/Visualiza-o-de-um-agregador-de-notas-para-animes/blob/main/anilist_1_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [ ]:


from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import pandas as pd
from pyspark.sql.types import *

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
spark = SparkSession.builder.config("spark.driver.memory", "8g").getOrCreate()

In [ ]:
pandas_df = pd.read_csv("/content/drive/MyDrive/data_anime/anilist_anime_data_complete1.csv", header=0)
anime_db = spark.createDataFrame(pandas_df)


print(anime_db.head())
anime_db.count()
anime_db.show()
anime_db.distinct().count()

"""#Limpando o Dataframe"""

anime_db = anime_db.drop('Unnamed: 0', 'id', 'idMal', 'updatedAt', 'nextAiringEpisode', 'airingSchedule')

anime_db.head()


"""#Tratando dados."""

frame_studio=anime_db.select('studios')

frame_studio.show()
frame_studio.printSchema()
frame_studio.count()
frame_studio=frame_studio.fillna('Vazio', subset=['studios'])
frame_studio.count()

"""#Separando os dados da coluna Studios, que possui dados semiestruturados em Json, então tive de separar os dados

"""

# O nome do estudio esta no campo NODE então fiz um inferschema para ele primeiro
node_schema = StructType([
    StructField('id', LongType(), True),
    StructField('name', StringType(), True),
    StructField('isAnimationStudio', BooleanType(), True)
])

#Agora definindo o inferschema da coluna como uma estrutura externa por campo Node
studio_object_schema = StructType([
    StructField('id', LongType(), True),
    StructField('isMain', BooleanType(), True),
    StructField('role', StringType(), True),
    StructField('node', node_schema, True) # 'node' is now a struct
])

# captando os dados do campo Studios e ai transformando no Array para
studios_array_schema = ArrayType(studio_object_schema)

# Agora aplicando From_json para transformar o campo em um Array
pedaco = frame_studio.withColumn('parsed_studios', from_json(col('studios'), studios_array_schema))

pedaco.show()

anime_db.printSchema()

"""### Extraindo e Filtrando o nome do estúdio de Animação
(Com auxilio de IA)
"""

from pyspark.sql.functions import col, from_json, when, lit, udf, get
from pyspark.sql.types import StringType

# Reusing schemas defined previously in cell BXNSWLY9PzJv
# node_schema, studio_object_schema, studios_array_schema

# 1. Parse o JSON da coluna 'studios' no DataFrame principal
anime_db = anime_db.withColumn(
    'parsed_studios',
    from_json(col('studios'), studios_array_schema)
)

# Define a UDF to extract the principal animation studio name
def get_principal_animation_studio(studios_list):
    if not studios_list:
        return 'Nulo'

    main_animation_studio_name = None
    first_animation_studio_name = None

    for studio_obj in studios_list:
        # Check if 'node' and 'isAnimationStudio' exist and is True
        # Accessing nested fields of a Row object
        if studio_obj and 'node' in studio_obj and studio_obj['node'] and 'isAnimationStudio' in studio_obj['node'] and studio_obj['node']['isAnimationStudio']:
            if 'isMain' in studio_obj and studio_obj['isMain']:
                # If a main animation studio is found, return its name immediately
                return studio_obj['node']['name']
            if first_animation_studio_name is None:
                # Store the first animation studio found (if no main one is found later)
                first_animation_studio_name = studio_obj['node']['name']

    # If no main animation studio was found, return the first animation studio's name
    if first_animation_studio_name:
        return first_animation_studio_name

    # If no animation studios were found at all
    return 'Sem estudio'

# Register the UDF
get_principal_animation_studio_udf = udf(get_principal_animation_studio, StringType())

# Apply the UDF to create the 'Estudio_Principal' column
anime_db = anime_db.withColumn(
    'Estudio',
    get_principal_animation_studio_udf(col('parsed_studios'))
)

# Remova as colunas intermediárias 'parsed_studios' e 'animation_studios_list'
anime_db = anime_db.drop('parsed_studios', 'studios')

# Exiba o DataFrame atualizado para verificar a nova coluna
anime_db.select('title_romaji', 'Estudio').show(truncate=False)
anime_db.count()

from pyspark.sql.types import StructType, StructField, LongType, StringType, BooleanType, ArrayType
from pyspark.sql.functions import col, from_json, concat_ws, array_distinct, expr # Import expr for transform if needed. Col.transform is available in recent versions.

# Define schema for a single tag object
# Renaming for clarity as it's for 'tags' column, not 'studios'
tag_object_schema = StructType([
    StructField('id', LongType(), True),
    StructField('name', StringType(), True),
    StructField('description', StringType(), True),
    StructField('category', StringType(), True),
    StructField('rank', LongType(), True),
    StructField('isGeneralSpoiler', BooleanType(), True),
    StructField('isMediaSpoiler', BooleanType(), True),
    StructField('isAdult', BooleanType(), True)
])

# Define the overall schema for the 'tags' column, which is an array of tag objects
tags_array_schema = ArrayType(tag_object_schema)

# Parse the JSON string in the 'tags' column into a structured array
anime_db = anime_db.withColumn('parsed_tags', from_json(col('tags'), tags_array_schema))

# Extract the 'name' from each tag object in the 'parsed_tags' array
# and collect them into a new array column 'tag_names_array'.
# Use a higher-order function like transform (available in Spark 3.1+)
# to apply a transformation to each element of the array.
# For older Spark versions, a UDF would be necessary.
# Colab typically runs on Spark 3.x, so transform should be available.

anime_db = anime_db.withColumn(
    'tag_names_array',
    expr("transform(parsed_tags, x -> x.name)") # Using expr with transform
)

# Convert the array of tag names into a comma-separated string
# Use array_distinct to ensure unique tags if desired.
anime_db = anime_db.withColumn('tags', concat_ws(', ', array_distinct(col('tag_names_array'))))

# Drop the intermediate parsing columns
anime_db = anime_db.drop('parsed_tags', 'tag_names_array')

# Show the updated DataFrame with the new 'tags' column
anime_db.select('title_romaji', 'tags').show(truncate=False)

anime_db.show()

# Optimized Tag expansion
tags_split = split(col("tags"), ", ")
for i in range(70):
    anime_db = anime_db.withColumn(f"Tags_{i+1}", get(tags_split, lit(i)))

anime_db=anime_db.withColumnRenamed('title_romaji','Titulo')
anime_db=anime_db.withColumnRenamed('title_english','Titulo_Ingles')
anime_db=anime_db.withColumnRenamed('type','Tipo')
anime_db=anime_db.withColumnRenamed('popularity','Popularidade')
anime_db=anime_db.withColumnRenamed('season','Temporada')
anime_db=anime_db.withColumnRenamed('seasonYear','Ano da Temporada')
anime_db=anime_db.withColumnRenamed('Source','Material de Origem')
anime_db=anime_db.withColumnRenamed('startDate_month','Mês de estreia')
anime_db=anime_db.withColumnRenamed('startDate_year','Ano de estreia')
anime_db=anime_db.withColumnRenamed('startDate_day','Dia de estreia')
anime_db=anime_db.withColumnRenamed('endDate_month','Mês de encerramento')
anime_db=anime_db.withColumnRenamed('endDate_year','Ano de encerramento')
anime_db=anime_db.withColumnRenamed('endDate_day','Dia de encerramento')
anime_db=anime_db.withColumnRenamed('averageScore','Nota Media')
anime_db=anime_db.withColumnRenamed('meanScore','Media de avaliação')
anime_db=anime_db.withColumnRenamed('countryOfOrigin','País')
anime_db=anime_db.withColumnRenamed('episodes','Episodios')
anime_db=anime_db.withColumnRenamed('isAdult','É Adulto')

# Specify columns for fillna to avoid type conflicts with strings
numeric_cols = ['Ano da Temporada', 'Ano de estreia', 'Mês de estreia', 'Dia de estreia', 'Nota Media', 'Popularidade', 'Mês de encerramento', 'Ano de encerramento', 'Dia de encerramento', 'Episodios', 'Media de avaliação']
anime_db = anime_db.fillna(0, subset=numeric_cols)

#Mais um campo Json para tratar
genres_schema = ArrayType(StringType())
anime_db = anime_db.withColumn('Genero', from_json(col('genres'), genres_schema))

# pegar a coluna, transformada em Array, e juntar numa string só, usando o comando Concat_ws, separando pela virgula
anime_db = anime_db.withColumn('Genero divição', concat_ws(',', col('Genero')))

for i in range(7):
    anime_db = anime_db.withColumn(f'Genero dividir_{i+1}', get(split(col('Genero divição'), ','), lit(i)))
# anime_db.show()
print(anime_db)
temporada_pt={'WINTER':'Inverno','SPRING':'Primavera','SUMMER':'Verão','FALL':'Outono', 'NaN':'Nulo'}
Tipo_pt={'TV':'TV','MOVIE':'Filme','SPECIAL':'Especial','OVA':'OVA','ONA':'Streaming/Internet','MUSIC':'Música'}
status_pt={'FINISHED':'Finalizado','RELEASING':'Lançando','NOT_YET_RELEASED':'Não lançado','CANCELLED':'Cancelado','HIATUS':'Hiato'}
pais_pt={'CN':'China','JP':'Japão', 'KR':'Coreia','TW':'Taiwan'}
media_pt={'ORIGINAL':'Original','MANGA':'Manga','LIGHT_NOVEL':'Light Novel','VISUAL_NOVEL':'Visual Novel','VIDEO_GAME':'Video Game','OTHER':'Outro','NOVEL':'Novel','DOUJINSHI':'Doujinshi','ANIME':'Anime','WEB_NOVEL':'Web Novel','LIVE_ACTION':'Live Action','GAME':'Game','COMIC':'Comédia','MULTIMEDIA_PROJECT':'Projeto Multimídia','PICTURE_BOOK':'Livro de Desenho', 'NaN':'Não declarado'}
anime_db = anime_db.replace(to_replace=temporada_pt, subset=['Temporada'])
# anime_db = anime_db.replace(to_replace=Tipo_pt, subset=['format'])
# anime_db = anime_db.replace(to_replace=status_pt, subset=['status'])
anime_db = anime_db.replace(to_replace=pais_pt, subset=['País'])
anime_db = anime_db.replace(to_replace=media_pt, subset=['Material de Origem'])
anime_db=anime_db.drop('Genero','Genero divição','genres','Unnamed: 0')
anime_db=anime_db.dropna(subset=['Titulo'])

anime_db.describe()

anime_db = anime_db.drop('Genero')
fatura=anime_db.select('genero dividir_1').distinct()
fatura.head(20)

#Fazendo contas com a variavel

def fazendo_calculos(column_name, mapping_dict):
    exp_mappin= lit(None) # Iniciar com valor nulo
    for gen_str, num_val in mapping_dict.items():
        exp_mappin = when(col(column_name) == gen_str, num_val).otherwise(exp_mappin)
    return exp_mappin


# anime_db.show()

anime_db = anime_db.withColumn('Mês de estreia', col('Mês de estreia').cast(IntegerType()))
anime_db = anime_db.withColumn('Ano de estreia', col('Ano de estreia').cast(IntegerType()))
anime_db = anime_db.withColumn('Dia de estreia', col('Dia de estreia').cast(IntegerType()))
anime_db = anime_db.withColumn('Mês de encerramento', col('Mês de encerramento').cast(IntegerType()))
anime_db = anime_db.withColumn('Ano de encerramento', col('Ano de encerramento').cast(IntegerType()))
anime_db = anime_db.withColumn('Dia de encerramento', col('Dia de encerramento').cast(IntegerType()))
anime_db = anime_db.withColumn('Ano da Temporada', col('Ano da Temporada').cast(IntegerType()))
anime_db = anime_db.withColumn('Episodios', col('Episodios').cast(IntegerType()))
anime_db = anime_db.withColumn('Popularidade', col('Popularidade').cast(IntegerType()))
anime_db = anime_db.withColumn('Media de avaliação', col('Media de avaliação').cast(IntegerType()))
anime_db = anime_db.withColumn('Nota Media', col('Nota Media').cast(IntegerType()))
# anime_db = anime_db.withColumn('duration', col('duration').cast(IntegerType()))
print(anime_db.printSchema())
anime_db.show()
anime_db.coalesce(1).write.csv('/content/drive/MyDrive/data_anime/mal.csv', header=True, mode='overwrite')

Row(Unnamed: 0=0, id=1497, idMal=1497.0, title_romaji='Aru Machi Kado no Monogatari', title_english='Tales of the Street Corner', type='ANIME', startDate_year=1962, startDate_month=11.0, startDate_day=5.0, endDate_year=1962.0, endDate_month=11.0, endDate_day=5.0, season='FALL', seasonYear=1962.0, episodes=1.0, countryOfOrigin='JP', source='ORIGINAL', updatedAt=1778693686, genres='["Drama", "Fantasy", "Music", "Romance"]', tags='[{"id": 341, "name": "No Dialogue", "description": "This work contains no dialogue.", "category": "Technical", "rank": 76, "isGeneralSpoiler": false, "isMediaSpoiler": false, "isAdult": false}, {"id": 111, "name": "War", "description": "Partly or completely set during wartime.", "category": "Theme-Other", "rank": 70, "isGeneralSpoiler": false, "isMediaSpoiler": true, "isAdult": false}, {"id": 595, "name": "Urban", "description": "Partly or completely set in a city.", "category": "Setting-Scene", "rank": 66, "isGeneralSpoiler": false, "isMediaSpoiler": false, "is